# Notebook 1: Analysis and Visualization of Spatial Transcriptomics Data

**Source:** [Scanpy Tutorial — Basic Spatial Analysis](https://scanpy-tutorials.readthedocs.io/en/latest/spatial/basic-analysis.html)  
**Author (tutorial):** Giovanni Palla  

This notebook demonstrates how to work with spatial transcriptomics data in Scanpy. We focus on 10x Genomics **Visium** data (Human Lymph Node) and also show a **MERFISH** example.

## Topics Covered
1. Reading Visium data
2. QC & preprocessing
3. Manifold embedding & Leiden clustering
4. Spatial visualization on H&E image
5. Cluster marker genes
6. MERFISH example

In [20]:
!pip install scanpy squidpy leidenalg python-igraph seaborn

## 0. Import Libraries

In [21]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

sc.logging.print_versions()
sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.verbosity = 3

/tmp/ipykernel_4786/78251993.py:8: FutureWarning: Use `print_header` instead
  sc.logging.print_versions()


## 1. Reading the Data

We use the **V1 Human Lymph Node** Visium dataset, publicly available from 10x Genomics.  
`sc.datasets.visium_sge()` downloads the data and returns an `AnnData` object containing counts, images, and spatial coordinates.

In [22]:
adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata.var_names_make_unique()

# Flag mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")

# Calculate QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

print(adata)

reading data/V1_Human_Lymph_Node/filtered_feature_bc_matrix.h5


/tmp/ipykernel_4786/2970670209.py:1: FutureWarning: Use `squidpy.datasets.visium` instead.
  adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")


 (0:00:04)


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1880: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 4035 × 36601
    obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'spatial'
    obsm: 'spatial'


The resulting `AnnData` object structure for Visium data includes:
- **`obs`**: per-spot metadata (in_tissue, array coordinates, QC metrics)
- **`var`**: per-gene metadata
- **`uns['spatial']`**: spatial metadata including images
- **`obsm['spatial']`**: spatial coordinates for each spot

## 2. QC and Preprocessing

We perform basic filtering of spots based on:
- Total counts (remove very low and very high count spots)
- Mitochondrial read percentage (remove dying/damaged cells)
- Gene detection (remove lowly expressed genes)

In [23]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize QC metric distributions
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])
axs[0].set_title("Total counts")

sns.histplot(
    adata.obs["total_counts"][adata.obs["total_counts"] < 10000],
    kde=False, bins=40, ax=axs[1]
)
axs[1].set_title("Total counts (<10,000)")

sns.histplot(adata.obs["n_genes_by_counts"], kde=False, bins=60, ax=axs[2])
axs[2].set_title("Genes by counts")

sns.histplot(
    adata.obs["n_genes_by_counts"][adata.obs["n_genes_by_counts"] < 4000],
    kde=False, bins=60, ax=axs[3]
)
axs[3].set_title("Genes by counts (<4,000)")

plt.tight_layout()
plt.show()

<Figure size 1200x320 with 4 Axes>

In [24]:
# Apply filters
sc.pp.filter_cells(adata, min_counts=5000)
sc.pp.filter_cells(adata, max_counts=35000)
adata = adata[adata.obs["pct_counts_mt"] < 20].copy()
print(f"Number of spots after MT filter: {adata.n_obs}")
sc.pp.filter_genes(adata, min_cells=10)

filtered out 44 cells that have less than 5000 counts
filtered out 130 cells that have more than 35000 counts
Number of spots after MT filter: 3861
filtered out 16916 genes that are detected in less than 10 cells


In [25]:
# Normalize, log-transform, and find highly variable genes
sc.pp.normalize_total(adata, inplace=True)       # normalize to total counts per cell
sc.pp.log1p(adata)                                # log1p transform
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)

normalizing counts per cell
    finished (0:00:00)
extracting highly variable genes
    finished (0:00:01)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)


## 3. Manifold Embedding & Clustering

We follow the standard Scanpy clustering workflow:
1. PCA
2. Nearest-neighbor graph
3. UMAP embedding
4. Leiden community detection

In [26]:
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(
    adata,
    key_added="clusters",
    flavor="igraph",
    directed=False,
    n_iterations=2
)

computing PCA
    with n_comps=50
    finished (0:00:03)
computing neighbors
    using 'X_pca' with n_pcs = 50
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:00)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:07)
running Leiden clustering
    finished: found 10 clusters and added
    'clusters', the cluster labels (adata.obs, categorical) (0:00:00)


In [27]:
# UMAP visualization
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(
    adata,
    color=["total_counts", "n_genes_by_counts", "clusters"],
    wspace=0.4,
    save="_01_clusters.png"
)

/tmp/ipykernel_4786/3517945718.py:3: FutureWarning: Argument `save` is deprecated and will be removed in a future version. Use `sc.pl.plot(show=False).figure.savefig()` instead.
  sc.pl.umap(


<Figure size 1344x320 with 5 Axes>

## 4. Visualization in Spatial Coordinates

`sc.pl.spatial()` overlays spot data on top of the H&E tissue image.  
Key parameters:
- `img_key`: which resolution image to use (`"hires"` or `"lowres"`)
- `crop_coord`: crop to a region of interest `(left, right, top, bottom)`
- `alpha_img`: transparency of the tissue image
- `bw`: convert image to grayscale
- `size`: scaling factor for spot sizes

In [28]:
plt.rcParams["figure.figsize"] = (8, 8)

# Spatial QC metrics
sc.pl.spatial(
    adata,
    img_key="hires",
    color=["total_counts", "n_genes_by_counts"]
)

/tmp/ipykernel_4786/3656403211.py:4: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


<Figure size 1425.6x640 with 4 Axes>

In [29]:
# Spatial cluster visualization
sc.pl.spatial(adata, img_key="hires", color="clusters", size=1.5)

/tmp/ipykernel_4786/1726887461.py:2: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, img_key="hires", color="clusters", size=1.5)


<Figure size 640x640 with 1 Axes>

In [30]:
# Zoom into specific clusters
# Here we look at clusters 5 and 9 cropped to a region of interest
sc.pl.spatial(
    adata,
    img_key="hires",
    color="clusters",
    groups=["5", "9"],
    crop_coord=[7000, 10000, 0, 6000],
    alpha=0.5,
    size=1.3
)

/tmp/ipykernel_4786/945654978.py:3: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


<Figure size 640x640 with 1 Axes>

## 5. Cluster Marker Genes

We compute differentially expressed marker genes for each cluster using a t-test.  
Then we visualize the top 10 markers for a cluster of interest using a heatmap.

In [31]:
# Rank genes by cluster
sc.tl.rank_genes_groups(adata, "clusters", method="t-test")

# Heatmap of top 10 marker genes for cluster 9
sc.pl.rank_genes_groups_heatmap(adata, groups="9", n_genes=10, groupby="clusters")

ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:02)
    using 'X_pca' with n_pcs = 50
Storing dendrogram info using `.uns['dendrogram_clusters']`
categories: 0, 1, 2, etc.
var_group_labels: 9


<Figure size 336x480 with 5 Axes>

In [32]:
# Visualize top markers in spatial context
# CR2 is a top marker for cluster 9 (B-cell follicle marker)
sc.pl.spatial(adata, img_key="hires", color=["clusters", "CR2"])

/tmp/ipykernel_4786/449936255.py:3: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, img_key="hires", color=["clusters", "CR2"])


<Figure size 1425.6x640 with 3 Axes>

In [33]:
# Other spatially-informative genes
# COL1A2: collagen (stromal), SYPL1: synaptoporin
sc.pl.spatial(adata, img_key="hires", color=["COL1A2", "SYPL1"], alpha=0.7)

/tmp/ipykernel_4786/1807452171.py:3: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, img_key="hires", color=["COL1A2", "SYPL1"], alpha=0.7)


<Figure size 1425.6x640 with 4 Axes>

## 6. MERFISH Example

For FISH-based spatial data (MERFISH, seqFISH), there is no tissue image. Instead, spatial coordinates are stored directly in `adata.obsm['spatial']`.

Dataset: U2-OS cells from [Xia et al. 2019 (PNAS)](https://www.pnas.org/content/116/39/19490.abstract).

> **Note:** Download the data files manually before running this section:
> - [Coordinates (xlsx)](https://www.pnas.org/doi/suppl/10.1073/pnas.1912459116/suppl_file/pnas.1912459116.sd15.xlsx) → `../data/pnas.1912459116.sd15.xlsx`
> - [Counts (csv)](https://www.pnas.org/doi/suppl/10.1073/pnas.1912459116/suppl_file/pnas.1912459116.sd12.csv) → `../data/pnas.1912459116.sd12.csv`

In [34]:
import os

coord_path = "../data/pnas.1912459116.sd15.xlsx"
counts_path = "../data/pnas.1912459116.sd12.csv"

if os.path.exists(coord_path) and os.path.exists(counts_path):
    # Load MERFISH coordinates and counts
    coordinates = pd.read_excel(coord_path, index_col=0)
    counts = sc.read_csv(counts_path).transpose()

    # Build AnnData — assign spatial coordinates
    adata_merfish = counts[coordinates.index, :].copy()
    adata_merfish.obsm["spatial"] = coordinates.to_numpy()

    # Standard preprocessing
    sc.pp.normalize_per_cell(adata_merfish, counts_per_cell_after=1e6)
    sc.pp.log1p(adata_merfish)
    sc.pp.pca(adata_merfish, n_comps=15)
    sc.pp.neighbors(adata_merfish)
    sc.tl.umap(adata_merfish)
    sc.tl.leiden(
        adata_merfish,
        key_added="clusters",
        resolution=0.5,
        n_iterations=2,
        flavor="igraph",
        directed=False
    )

    print(adata_merfish)

    # UMAP + Spatial visualization
    sc.pl.umap(adata_merfish, color="clusters")
    sc.pl.embedding(adata_merfish, basis="spatial", color="clusters")
else:
    print("MERFISH data files not found. Please download them to ../data/ and re-run.")

normalizing by total count per cell
    finished (0:00:00): normalized adata.X and added
    'n_counts', counts per cell before normalization (adata.obs)
computing PCA
    with n_comps=15


/tmp/ipykernel_4786/4107757478.py:16: FutureWarning: Use `sc.pp.normalize_total` instead.
  sc.pp.normalize_per_cell(adata_merfish, counts_per_cell_after=1e6)


    finished (0:00:00)
computing neighbors
    using 'X_pca' with n_pcs = 15
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:00)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:01)
running Leiden clustering
    finished: found 7 clusters and added
    'clusters', the cluster labels (adata.obs, categorical) (0:00:00)
AnnData object with n_obs × n_vars = 645 × 12903
    obs: 'n_counts', 'clusters'
    uns: 'log1p', 'pca', 'neighbors', 'umap', 'clusters'
    obsm: 'spatial', 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'


<Figure size 640x640 with 1 Axes>

<Figure size 640x640 with 1 Axes>

## Summary

In this notebook we learned:
- How to load Visium data and inspect the AnnData structure
- QC filtering based on counts, genes, and mitochondrial percentage
- Standard normalization → HVG → PCA → UMAP → Leiden clustering
- Spatial visualization overlaid on H&E tissue images
- Zooming into regions of interest and adjusting transparency
- Finding and visualizing cluster marker genes spatially
- How MERFISH data is structured differently (coordinates in `obsm` instead of images)
